# LightOnOCR-2 Magyar Fine-tuning (v8)

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Telepítés
!pip install -q transformers>=4.45.0 peft datasets accelerate pillow opencv-python-headless

# Fontok telepítése
!apt-get update -qq
!apt-get install -qq fonts-dejavu-core fonts-dejavu-extra fonts-liberation fonts-freefont-ttf
!fc-cache -f
print('Telepítés kész')

In [ ]:
# 2. Font keresés - részletes debug
import os
import glob
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# Ismert font útvonalak Colab-on
FONT_DIRS = [
    '/usr/share/fonts/truetype/dejavu',
    '/usr/share/fonts/truetype/liberation',
    '/usr/share/fonts/truetype/freefont',
    '/usr/share/fonts/truetype',
]

print('=== Font könyvtárak ellenőrzése ===')
for d in FONT_DIRS:
    if os.path.exists(d):
        files = glob.glob(f'{d}/*.ttf')
        print(f'{d}: {len(files)} ttf fájl')
        for f in files[:3]:
            print(f'  - {os.path.basename(f)}')
    else:
        print(f'{d}: NEM LÉTEZIK')

In [ ]:
# 3. Font teszt függvény
def test_font(font_path, test_text='őűŐŰ'):
    """Teszteli, hogy a font rendereli-e a magyar karaktereket"""
    try:
        font = ImageFont.truetype(font_path, 32)
        img = Image.new('RGB', (150, 50), 'white')
        draw = ImageDraw.Draw(img)
        draw.text((10, 10), test_text, fill='black', font=font)
        
        # Fekete pixelek száma
        arr = np.array(img)
        black = np.sum(arr < 100)
        return black > 50, black, img
    except Exception as e:
        return False, 0, None

# Összes TTF font keresése
all_fonts = []
for d in FONT_DIRS:
    all_fonts.extend(glob.glob(f'{d}/**/*.ttf', recursive=True))
    all_fonts.extend(glob.glob(f'{d}/*.ttf'))

all_fonts = list(set(all_fonts))
print(f'\nÖsszes talált font: {len(all_fonts)}')

# Tesztelés
print('\n=== Font tesztek ===')
FONTS = []
from IPython.display import display

for fp in sorted(all_fonts):
    name = os.path.basename(fp)
    ok, pixels, img = test_font(fp)
    status = '✓' if ok else '✗'
    print(f'{status} {name}: {pixels} fekete pixel')
    if ok:
        FONTS.append((name.replace('.ttf', ''), fp))
        if len(FONTS) <= 3:
            display(img)

print(f'\n=== {len(FONTS)} működő font ===')

In [ ]:
# 4. Ha nincs elég font, töltsük le kézzel
if len(FONTS) < 3:
    print('Kevés font, DejaVu letöltése...')
    !mkdir -p /content/fonts
    !wget -q 'https://github.com/dejavu-fonts/dejavu-fonts/releases/download/version_2_37/dejavu-fonts-ttf-2.37.zip' -O /tmp/dejavu.zip
    !unzip -q -o /tmp/dejavu.zip -d /tmp/
    !cp /tmp/dejavu-fonts-ttf-2.37/ttf/*.ttf /content/fonts/
    !ls /content/fonts/
    
    # Újra keresés
    FONTS = []
    for fp in glob.glob('/content/fonts/*.ttf'):
        ok, pixels, img = test_font(fp)
        if ok:
            name = os.path.basename(fp).replace('.ttf', '')
            FONTS.append((name, fp))
            print(f'✓ {name}')
    
    print(f'\n=== {len(FONTS)} font a letöltésből ===')
else:
    print(f'OK: {len(FONTS)} font elérhető')

In [ ]:
# 5. Augmentációk
import cv2
import random
from PIL import ImageFilter
from pathlib import Path

def add_noise(img, intensity=0.02):
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, intensity * 255, arr.shape)
    return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

def add_salt_pepper(img, amount=0.005):
    arr = np.array(img)
    salt = np.random.random(arr.shape[:2]) < amount/2
    arr[salt] = 255
    pepper = np.random.random(arr.shape[:2]) < amount/2
    arr[pepper] = 0
    return Image.fromarray(arr)

def rotate_image(img, max_angle=2.0):
    return img.rotate(random.uniform(-max_angle, max_angle), fillcolor='white')

def perspective_transform(img, intensity=0.02):
    arr = np.array(img)
    h, w = arr.shape[:2]
    src = np.float32([[0,0], [w,0], [w,h], [0,h]])
    o = int(min(w,h) * intensity)
    dst = np.float32([[random.randint(0,o), random.randint(0,o)],
                      [w-random.randint(0,o), random.randint(0,o)],
                      [w-random.randint(0,o), h-random.randint(0,o)],
                      [random.randint(0,o), h-random.randint(0,o)]])
    M = cv2.getPerspectiveTransform(src, dst)
    return Image.fromarray(cv2.warpPerspective(arr, M, (w, h), borderValue=(255,255,255)))

def apply_aug(img):
    if random.random() < 0.3: img = add_noise(img, random.uniform(0.01, 0.03))
    if random.random() < 0.2: img = add_salt_pepper(img, random.uniform(0.002, 0.01))
    if random.random() < 0.4: img = rotate_image(img, random.uniform(0.5, 2.5))
    if random.random() < 0.2: img = perspective_transform(img, random.uniform(0.01, 0.03))
    if random.random() < 0.2: img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 0.8)))
    return img

print('✓ Augmentációk')

In [ ]:
# 6. Adatgenerálás
import json

WORDS = [
    'őr', 'őriz', 'ők', 'ősz', 'ősi', 'őszinte', 'őrült',
    'erő', 'idő', 'mező', 'tető', 'fő', 'nő', 'bő', 'hő',
    'belső', 'külső', 'felső', 'alsó', 'utolsó', 'első',
    'költő', 'festő', 'vezető', 'börtön', 'könyv', 'között',
    'Győr', 'dőlt', 'dől', 'töröl', 'pörög', 'görög', 'örök',
    'űr', 'űrlap', 'gyűrű', 'tűz', 'fűz', 'gyűjt', 'gyűlés',
    'tűnik', 'fűszer', 'hűtő', 'hűvös', 'hűség',
    'szürke', 'szűk', 'szűr', 'sűrű', 'bűvös', 'működik', 'műszer',
    'Csatornadíj', 'vízdíj', 'díj', 'tükörfúrógép', 'árvíztűrő',
    'fizetendő', 'összeg', 'adószám', 'határidő',
]

def gen_text():
    lines = [' '.join(random.sample(WORDS, random.randint(5, 8)))]
    lines.append(f'Fizetendő összeg: {random.randint(1,99)} {random.randint(100,999):03d} Ft')
    lines.append(f'Csatornadíj: {random.randint(1,9)} {random.randint(100,999):03d} Ft')
    lines.append(f'Adószám: {random.randint(10000000,99999999)}-{random.randint(1,2)}-{random.randint(10,99)}')
    lines.append('öüóőúéáűí - ÖÜÓŐÚÉÁŰÍ')
    lines.append('Árvíztűrő tükörfúrógép')
    return '\n'.join(lines)

def render(text, font_path, size=24):
    font = ImageFont.truetype(font_path, size)
    lines = text.split('\n')
    h = len(lines) * int(size * 1.5) + 80
    img = Image.new('RGB', (850, h), random.choice(['white', '#fafafa', '#f5f5f5']))
    draw = ImageDraw.Draw(img)
    y = 40
    for line in lines:
        draw.text((40, y), line, fill='black', font=font)
        y += int(size * 1.5)
    return img

assert len(FONTS) >= 1, 'Nincs működő font!'

Path('training_data/images').mkdir(parents=True, exist_ok=True)
annotations = []
N = 800

print(f'Generálás: {N} kép, {len(FONTS)} fonttal...')
for i in range(N):
    text = gen_text()
    fname, fpath = random.choice(FONTS)
    img = render(text, fpath, random.choice([18,20,22,24,26,28]))
    aug = random.random() < 0.6
    if aug: img = apply_aug(img)
    img.save(f'training_data/images/{i:05d}.png')
    annotations.append({'image': f'{i:05d}.png', 'text': text, 'font': fname, 'aug': aug})
    if (i+1) % 100 == 0: print(f'  {i+1}/{N}')

with open('training_data/annotations.jsonl', 'w', encoding='utf-8') as f:
    for a in annotations:
        f.write(json.dumps(a, ensure_ascii=False) + '\n')

print(f'\n✓ {N} kép')

In [ ]:
# 7. Példák
print('Példák (ellenőrizd az ő és ű karaktereket!):')
for i in [0, 100, 200]:
    print(f'\nKép #{i}:')
    display(Image.open(f'training_data/images/{i:05d}.png'))

In [ ]:
# 8. Modell
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

MODEL_ID = 'lightonai/LightOnOCR-2-1B-base'
print(f'Modell: {MODEL_ID}')

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
processor = AutoProcessor.from_pretrained(MODEL_ID)

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], lora_dropout=0.05)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 9. Dataset
from datasets import Dataset

def load_data():
    data = []
    with open('training_data/annotations.jsonl', encoding='utf-8') as f:
        for line in f:
            e = json.loads(line)
            data.append({'image_path': f"training_data/images/{e['image']}", 'text': e['text']})
    return Dataset.from_list(data)

def process(ex):
    img = Image.open(ex['image_path']).convert('RGB')
    img_in = processor.image_processor(img, return_tensors='pt')
    txt_in = processor.tokenizer(ex['text'], return_tensors='pt', padding='max_length', max_length=512, truncation=True)
    return {
        'pixel_values': img_in['pixel_values'].squeeze(0),
        'input_ids': txt_in['input_ids'].squeeze(0),
        'attention_mask': txt_in['attention_mask'].squeeze(0),
        'labels': txt_in['input_ids'].squeeze(0),
    }

dataset = load_data().map(process, remove_columns=['image_path', 'text'])
print(f'✓ {len(dataset)} kép')

In [ ]:
# 10. Training
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir='./lighton-hun-lora',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    logging_steps=25,
    save_steps=150,
    bf16=True,
    remove_unused_columns=False,
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=dataset)
print('Tanítás indul...')
trainer.train()
print('\n✓ Kész!')

In [ ]:
# 11. Mentés
model.save_pretrained('./lighton-hun-lora')
merged = model.merge_and_unload()
merged.save_pretrained('./lighton-hun-merged')
processor.save_pretrained('./lighton-hun-merged')
print('✓ Mentve')

In [ ]:
# 12. Teszt
print('Teszt:')
for idx in [0, 200, 400]:
    img = Image.open(f'training_data/images/{idx:05d}.png')
    inputs = processor.image_processor(img, return_tensors='pt')
    inputs = {k: v.to(merged.device) for k, v in inputs.items()}
    inputs['input_ids'] = processor.tokenizer('', return_tensors='pt')['input_ids'].to(merged.device)
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=400, do_sample=False)
    print(f'\n=== #{idx} ===')
    display(img)
    print(processor.tokenizer.decode(out[0], skip_special_tokens=True)[:300])

In [ ]:
# 13. Letöltés
!zip -r lighton-hun-merged.zip lighton-hun-merged/
from google.colab import files
files.download('lighton-hun-merged.zip')
print('\nMAC: unzip && mlx_vlm convert --hf-path lighton-hun-merged --mlx-path models/lighton-hun-mlx -q --q-bits 4')